# Build Enriched Layer

### Mini challenge
**Zkus načíst data ze souborů do DataFrames, které jsou dále použity níže v kódu.**

In [ ]:
df_wines = "doplň kód"
df_sales = "doplň kód"
df_reviews = "doplň kód"

### Ověření, že jsi správně načrtl data

In [ ]:
display(df_wines.limit(3))
display(df_sales.limit(3))
display(df_reviews.limit(3))

## Schema for Enriched layer

In [6]:
%%sql
CREATE SCHEMA IF NOT EXISTS Enriched COMMENT 'Cleaned, structured, enriched data'

StatementMeta(, 1b9c57f8-6129-4b51-947b-0c5a85c0ca26, 8, Finished, Available, Finished, False)

<Spark SQL result set with 0 rows and 0 fields>

## Wines

In [ ]:
from pyspark.sql.functions import col, when

# Change datatypes
df_wines = df_wines.withColumn("UnitPrice",col("UnitPrice").cast("decimal(10,2)"))
df_wines = df_wines.withColumn("WineId",col("WineId").cast("integer"))
df_wines = df_wines.withColumn("Vintage",col("Vintage").cast("string"))

# Write 
df_wines.write.format("delta").mode("overwrite").option("mergeSchema", "true").saveAsTable("Enriched.Wines")

# Display data
display(df_wines.limit(5))

print(f"Written {df_wines.count()} rows.")

## Sales

In [ ]:
from pyspark.sql.functions import col, to_date

# Change datatypes
df_sales = df_sales.withColumn("Discount",col("Discount").cast("decimal(10,2)"))
df_sales = df_sales.withColumn("Quantity",col("Quantity").cast("integer"))
df_sales = df_sales.withColumn("SalesId",col("SalesId").cast("integer"))
df_sales = df_sales.withColumn("WineId",col("WineId").cast("integer"))
df_sales = df_sales.withColumn("Date",to_date(col("Date"), "dd.MM.yyyy"))

# Display data
display(df_sales.limit(5))

In [ ]:
from pyspark.sql.functions import lit, col

# Enrich with df_wines & Country
df_sales_enriched = df_sales.join(
    df_wines.select("WineId", "UnitPrice"), 
    "WineId", 
    "left"
)

# Add default Country
df_result = df_sales_enriched.withColumn("Country", lit("Česká republika"))

# Select columns
df_result = df_result.select("SalesId", "Date", "PaymentMethod", "Store", "WineId", "UnitPrice", "Quantity", "Discount", "Country") #"TotalAmount")

# Write 
df_result.write.format("delta").mode("overwrite").option("mergeSchema", "true").saveAsTable("Enriched.Sales")

# Display data
display(df_result.limit(5))

print(f"Written {df_result.count()} rows.")

## Reviews

In [ ]:
from pyspark.sql.functions import col, to_date

# Change datatypes
df_reviews = df_reviews.withColumn("ReviewId",col("ReviewId").cast("integer"))
df_reviews = df_reviews.withColumn("Date",to_date(col("Date"), "dd.MM.yyyy"))

# Display data
display(df_reviews.limit(5))

In [ ]:
# INNER JOIN - for existing wines only
df_wine_reviews = df_reviews.join(
    df_wines.select("WineCode"), 
    "WineCode", 
    "inner"
)

# Write 
df_wine_reviews.write.format("delta").mode("overwrite").option("mergeSchema", "true").saveAsTable("Enriched.Reviews")

# Display data
display(df_wine_reviews.limit(5))

print(f"Written {df_wine_reviews.count()} rows.")